# R3 · Smoke test `grammarly/coedit-large` (Tuần 1)

Mục tiêu (Plan4project A4, A5, C1, C4, C6):
1. Xác nhận load thành công `grammarly/coedit-large` (`AutoModelForSeq2SeqLM` / `T5ForConditionalGeneration` + `AutoTokenizer`) trên GPU T4 (Colab/Kaggle) hoặc GPU cá nhân.
2. Kiểm chứng tính ổn định số học ở `float32` / `bfloat16` (tránh lỗi tràn số `NaN` của kiến trúc T5 khi ép sang `float16` trên T4).
3. Chạy thử với tiền tố chuẩn `"Fix grammatical errors in this sentence: "` ở chế độ greedy (`do_sample=False, num_beams=1, max_new_tokens=256`) trên 6 dòng báo cáo tài chính tiếng Anh tự soạn (lỗi ký tự OCR `rn/m`, lỗi số `1,25O` / `l5%`, tách/gộp từ, và dòng đã đúng).
4. Đo VRAM và tốc độ (`s/dòng`) để ước tính thời gian chạy full `en_test.jsonl` (500 dòng × 2 mức nhiễu = 1.000 dòng).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U transformers accelerate sentencepiece
import sys, time, unicodedata, torch, transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available(), "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Load checkpoint `grammarly/coedit-large` với `dtype` an toàn
- Trên GPU Ampere+ (`compute_capability >= 8.0`, vd RTX 30xx/40xx, L4, A100): dùng `torch.bfloat16` (~1.6 GB VRAM).
- Trên GPU Tesla T4 (`compute_capability == 7.5`): dùng **`torch.float32`** (~3.1 GB VRAM, vừa thoải mái trên T4 16 GB và tránh hoàn toàn lỗi tràn số `NaN` của T5 ở `float16`).

In [ ]:
MODEL_ID = "grammarly/coedit-large"
PREFIX = "Fix grammatical errors in this sentence: "

if torch.cuda.is_available() and torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability()[0] >= 8:
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float32

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto")
except Exception:
    try:
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE)
    except TypeError:
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    if torch.cuda.is_available():
        model = model.to("cuda")
model.eval()

vram_gb = torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
print(f"Loaded {type(model).__name__} in {time.time() - t0:.1f}s | dtype={DTYPE} | VRAM={vram_gb:.2f} GB")

## 2. Chạy thử trên 6 câu báo cáo tài chính tiếng Anh mô phỏng lỗi OCR

In [ ]:
SAMPLES = [
    ("KT (rn->m)", "The Cornpany's owners' equity increased cornpared to the prior year.", "The Company's owners' equity increased compared to the prior year."),
    ("SO (O->0, l->1)", "Profit after tax reached VND 1,25O billion, up l5% year on year.", "Profit after tax reached VND 1,250 billion, up 15% year on year."),
    ("TACH (split/merge)", "Cash and cashequivalents at year end amounted to VND 450 billion.", "Cash and cash equivalents at year end amounted to VND 450 billion."),
    ("MAT (missing char)", "Consolidated financia statements are prepared under IFRS.", "Consolidated financial statements are prepared under IFRS."),
    ("CLEAN (số liệu đúng)", "Cost of goods sold decreased by 8% compared to 2024.", "Cost of goods sold decreased by 8% compared to 2024."),
    ("CLEAN (thuật ngữ)", "Retained earnings and EBITDA remained positive in Q4 2025.", "Retained earnings and EBITDA remained positive in Q4 2025."),
]

exact = 0
t_start = time.time()
for tag, src, tgt in SAMPLES:
    inp = f"{PREFIX}{src.strip()}"
    enc = tokenizer(inp, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=256, do_sample=False, num_beams=1)
    pred = unicodedata.normalize("NFC", tokenizer.decode(out[0], skip_special_tokens=True).strip())
    ok = (pred == tgt)
    exact += int(ok)
    print(f"[{tag}] match={ok}")
    print(f"  SRC : {src}")
    print(f"  PRED: {pred}")
    print(f"  TGT : {tgt}\n")

elapsed_b1 = time.time() - t_start
sec_b1 = elapsed_b1 / len(SAMPLES)

# Đo tốc độ thực tế theo batch (batch_size=6, tương đương batch inference thật)
batch_inputs = [f"{PREFIX}{src.strip()}" for _, src, _ in SAMPLES]
enc_b = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
if torch.cuda.is_available():
    torch.cuda.synchronize()
t_b = time.time()
with torch.inference_mode():
    _ = model.generate(**enc_b, max_new_tokens=256, do_sample=False, num_beams=1)
if torch.cuda.is_available():
    torch.cuda.synchronize()
sec_batch = (time.time() - t_b) / len(SAMPLES)

peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
print("=== TỔNG KẾT SMOKE TEST R3 ===")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | dtype: {DTYPE}")
print(f"Khớp hoàn toàn target: {exact}/{len(SAMPLES)} ({exact / len(SAMPLES) * 100:.1f}%)")
print(f"Tốc độ batch=1: {sec_b1:.3f} s/dòng | Tốc độ batch={len(SAMPLES)}: {sec_batch:.3f} s/dòng")
print(f"Ước tính chạy full 1.000 dòng en_test (batch={len(SAMPLES)}): ~{1000 * sec_batch / 60:.1f} phút")
print(f"VRAM đỉnh: {peak_vram:.2f} GB")